Scenario: Customer Support Chatbot Workflow
Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

- State Definition (BotState)
- The chatbot keeps track of:
- The question asked by the customer.
- The answer generated.
- The history of all past questions.
- Think of this as the chatbot’s notebook where it records the conversation.

- Nodes (Functions)
- get_answer:
When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
It also adds the question to the history log.
- format_output:
Before sending the response back to the customer, the chatbot reformats it into a friendly style:
“Bot says: Answer to: What are your store hours?”

- Graph Flow
- The chatbot starts at the get_answer node (entry point).
- Once the answer is generated, it flows to the format_output node.
- Finally, the conversation ends at END, meaning the chatbot has
 delivered its response.


In [3]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)



In [4]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
import os
from dotenv import load_dotenv

load_dotenv()

class BotState(TypedDict):
    question: str
    answer: str
    history: List[str]

HF_API_KEY = os.getenv("API_KEY_HUGG")

if not HF_API_KEY:
    raise ValueError("Please set HF_API_KEY in your .env file")

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {HF_API_KEY}",
    "Content-Type": "application/json"
}

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

def call_llm(messages):
    payload = {
        "model": MODEL_ID,
        "messages": messages,
        "max_tokens": 500,
        "temperature": 0.7
    }

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code != 200:
        try:
            error = response.json()
        except:
            error = response.text
        raise Exception(f"HF API Error {response.status_code}: {error}")

    result = response.json()

    if "choices" not in result:
        raise ValueError(f"Unexpected response: {result}")

    return result["choices"][0]["message"]["content"]

def get_answer(state: BotState):
    q = state["question"]

    messages = [
        {"role": "system", "content": "You are a helpful and professional customer support assistant."}
    ]

    for past_q in state["history"]:
        messages.append({"role": "user", "content": past_q})

    messages.append({"role": "user", "content": q})

    ans = call_llm(messages)

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {
        "answer": f"🤖 Bot says:\n{state['answer']}"
    }

graph = StateGraph(BotState)

graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

app = graph.compile()

if __name__ == "__main__":
    state = {
        "question": "",
        "answer": "",
        "history": []
    }

    print("💬 Customer Support Chatbot (type 'exit' to quit)\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("👋 Goodbye!")
            break

        state["question"] = user_input

        result = app.invoke(state)

        print(result["answer"])
        print("-" * 50)

        state["history"] = result["history"]

💬 Customer Support Chatbot (type 'exit' to quit)

🤖 Bot says:
I'm a customer support assistant. I'm here to help answer any questions you may have, provide information, and assist with any issues or concerns you might be experiencing. I'm a text-based AI assistant, which means I can communicate with you through text messages, emails, or chat windows like this one. I'm available 24/7 to help with any questions or problems you might have, so feel free to ask me anything!
--------------------------------------------------
🤖 Bot says:
It seems like you would like to end our conversation. I'll be happy to help you with any questions or concerns you had before you decided to exit. If you change your mind and need assistance in the future, feel free to come back and I'll be here to help.

If you have any other questions or concerns right now, I can assist you with that. Otherwise, have a great day!
--------------------------------------------------
👋 Goodbye!


Scenario: AI-Powered Study Assistant (Flashcard-Based)
1. State Definition
The assistant maintains a notebook-like state for each learner:
- topic → The subject the learner is studying (e.g., "Photosynthesis").
- flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
- progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

2. Workflow (Graph of States)
Each learner interaction flows through nodes until a flashcard is delivered:
- Input Node
- Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
- State updates: topic = "cell biology"
- Generation Node (Groq API)
- Groq generates a flashcard:
- flashcard.question = "What organelle is known as the powerhouse of the cell?"
- flashcard.answer = "Mitochondria"
- Response Node
- Assistant presents the flashcard question to the learner.
- Evaluation Node
- Learner responds with their answer.
- Assistant checks correctness and updates progress.
- History Node
- Logs the flashcard attempt:
- progress = [{question, learner_answer, correct/incorrect}]

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
import os
from dotenv import load_dotenv

load_dotenv()

class StudyState(TypedDict):
    topic: str
    question: str
    answer: str
    learner_answer: str
    feedback: str
    progress: List[Dict]

HF_API_KEY = os.getenv("API_KEY_HUGG")  # groq was not working

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {HF_API_KEY}",
    "Content-Type": "application/json"
}

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

def call_llm(messages):
    payload = {
        "model": MODEL_ID,
        "messages": messages,
        "max_tokens": 300,
        "temperature": 0.7
    }
    response = requests.post(API_URL, headers=headers, json=payload)
    if response.status_code != 200:
        raise Exception(response.text)
    return response.json()["choices"][0]["message"]["content"]

def generate_flashcard(state: StudyState):
    topic = state["topic"]

    prompt = f"Create one flashcard for topic '{topic}'. Return strictly in format:\nQuestion: ...\nAnswer: ..."

    output = call_llm([{"role": "user", "content": prompt}])

    lines = output.split("\n")
    q = ""
    a = ""

    for line in lines:
        if "Question:" in line:
            q = line.replace("Question:", "").strip()
        if "Answer:" in line:
            a = line.replace("Answer:", "").strip()

    return {
        "question": q,
        "answer": a
    }

def present_question(state: StudyState):
    print(f"\n📘 Question: {state['question']}")
    user_ans = input("Your Answer: ")
    return {"learner_answer": user_ans}

def evaluate_answer(state: StudyState):
    correct = state["answer"].lower().strip()
    user = state["learner_answer"].lower().strip()

    if correct in user or user in correct:
        feedback = "✅ Correct!"
        result = "correct"
    else:
        feedback = f"❌ Incorrect! Correct answer: {state['answer']}"
        result = "incorrect"

    return {
        "feedback": feedback,
        "progress": state["progress"] + [{
            "question": state["question"],
            "your_answer": state["learner_answer"],
            "correct_answer": state["answer"],
            "result": result
        }]
    }

def show_feedback(state: StudyState):
    print(state["feedback"])
    return {}

graph = StateGraph(StudyState)

graph.add_node("generate", generate_flashcard)
graph.add_node("ask", present_question)
graph.add_node("evaluate", evaluate_answer)
graph.add_node("feedback", show_feedback)

graph.set_entry_point("generate")
graph.add_edge("generate", "ask")
graph.add_edge("ask", "evaluate")
graph.add_edge("evaluate", "feedback")
graph.add_edge("feedback", END)

app = graph.compile()

if __name__ == "__main__":
    topic = input("Enter topic: ")

    state = {
        "topic": topic,
        "question": "",
        "answer": "",
        "learner_answer": "",
        "feedback": "",
        "progress": []
    }

    while True:
        app.invoke(state)
        cont = input("\nDo you want another question? (yes/no): ")
        if cont.lower() != "yes":
            break

    print("\n📊 Progress Summary:")
    for p in state["progress"]:
        print(p)


📘 Question: What is LLM?
❌ Incorrect! Correct answer: Large Language Model, a type of artificial intelligence that uses natural language processing to generate human-like text responses.

📘 Question: What is an LLM?
❌ Incorrect! Correct answer: An LLM stands for Large Language Model, which is a type of artificial intelligence designed to process and generate human-like language, often used in applications such as chatbots, virtual assistants, and natural language processing tasks.

📊 Progress Summary:


\Scenario: AI-Powered Project Tracker (Task-Based)
1. State Definition
The assistant maintains a notebook-like state for each project:
- task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
- status → The current state of the task (e.g., "in progress", "completed", "blocked").
- log → A history of all updates, including who made them and when.

2. Workflow (Graph of States)
Each project update flows through nodes until the task status is refreshed:
- Input Node
- Team member submits an update (e.g., "Report draft completed").
- State updates: task = "Q1 financial report"
- Processing Node (Groq API)
- Groq interprets the update and assigns a status:
- status = "completed"
- Response Node
- Assistant confirms the update back to the team:
- "Task Q1 financial report marked as completed."
- History Node
- Logs the update:
- log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]

In [7]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
import os
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    response: str
    log: List[Dict]

HF_API_KEY = os.getenv("API_KEY_HUGG")

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {HF_API_KEY}",
    "Content-Type": "application/json"
}

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

def call_llm(messages):
    payload = {
        "model": MODEL_ID,
        "messages": messages,
        "max_tokens": 200,
        "temperature": 0.3
    }
    response = requests.post(API_URL, headers=headers, json=payload)
    if response.status_code != 200:
        raise Exception(response.text)
    return response.json()["choices"][0]["message"]["content"]

def process_update(state: ProjectState):
    prompt = f"""
Extract task status from the update.
Possible statuses: completed, in progress, blocked.

Update: {state['update']}

Return only one word: completed / in progress / blocked
"""
    result = call_llm([{"role": "user", "content": prompt}]).lower()

    if "complete" in result:
        status = "completed"
    elif "block" in result:
        status = "blocked"
    else:
        status = "in progress"

    return {"status": status}

def generate_response(state: ProjectState):
    return {
        "response": f"✅ Task '{state['task']}' marked as {state['status']}."
    }

def update_log(state: ProjectState):
    entry = {
        "task": state["task"],
        "update": state["update"],
        "status": state["status"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    return {
        "log": state["log"] + [entry]
    }

graph = StateGraph(ProjectState)

graph.add_node("process", process_update)
graph.add_node("respond", generate_response)
graph.add_node("log", update_log)

graph.set_entry_point("process")
graph.add_edge("process", "respond")
graph.add_edge("respond", "log")
graph.add_edge("log", END)

app = graph.compile()

if __name__ == "__main__":
    state = {
        "task": "Q1 financial report",           # hardcoding only one
        "update": "",
        "status": "",
        "response": "",
        "log": []
    }

    print("📊 AI Project Tracker (type 'exit' to quit)\n")

    while True:
        user_update = input("Update: ")

        if user_update.lower() == "exit":
            break

        state["update"] = user_update

        result = app.invoke(state)

        print(result["response"])
        print("-" * 50)

        state["status"] = result["status"]
        state["log"] = result["log"]

    print("\n📜 Project Log:")
    for entry in state["log"]:
        print(entry)

📊 AI Project Tracker (type 'exit' to quit)

✅ Task 'Q1 financial report' marked as completed.
--------------------------------------------------
✅ Task 'Q1 financial report' marked as completed.
--------------------------------------------------

📜 Project Log:
{'task': 'Q1 financial report', 'update': 'completed', 'status': 'completed', 'timestamp': '2026-03-20 11:51:02'}
{'task': 'Q1 financial report', 'update': 'add task 2 do python project', 'status': 'completed', 'timestamp': '2026-03-20 11:51:40'}


In [4]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
import os
from dotenv import load_dotenv

load_dotenv()

class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

HF_API_KEY = os.getenv("API_KEY_HUGG")

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {HF_API_KEY}",
    "Content-Type": "application/json"
}

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

def hf_call(prompt):
    payload = {
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 400,
        "temperature": 0.7
    }

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code != 200:
        raise Exception(response.text)

    return response.json()["choices"][0]["message"]["content"]

def draft_email(state: EmailState):
    draft = hf_call(f"Write a professional email about: {state['task']}")
    print(f"\n📝 Draft:\n{draft}\n")
    return {"draft": draft}

def human_review(state: EmailState):
    print("📧 Waiting for approval...\n")
    return {}

def send_email(state: EmailState):
    if state.get("approved", False):
        print("✅ Email sent")
        return {"task": f"SENT: {state['task']}"}
    else:
        print("❌ Email rejected")
        return {"task": f"REJECTED: {state['task']}"}

g = StateGraph(EmailState)

g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)

checkpointer = MemorySaver()

app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)

thread = {"configurable": {"thread_id": "email-1"}}
task = input("Enter topic of email")
app.invoke({"task": task, "draft": "", "approved": False}, thread)

decision = input("Approve email? (yes/no): ").lower()

approved = True if decision == "yes" else False

app.invoke({"approved": approved}, thread)


📝 Draft:
Subject: Dress Code and Timing Review

Dear [Manager's Name/Superior's Name],

I hope this email finds you well. I am writing to bring to your attention two aspects that have been causing some inconvenience and discussion among the team: the current dress code and the timing of our work hours.

Regarding the dress code, I believe it is essential to establish a clear and consistent policy that reflects our company's culture and values. The current dress code is somewhat ambiguous, leading to varying interpretations among team members. To address this issue, I suggest we implement a more defined dress code policy that is inclusive and respectful of individual preferences while maintaining a professional atmosphere.

Some possible suggestions for consideration include:

1. Introducing a more standard dress code for workplace attire, which would include guidelines for tops, bottoms, shoes, and accessories.
2. Creating a dress code policy document that outlines expectations and co

{'task': 'Fix the dress code and timing',
 'draft': "Subject: Revision to Company Dress Code and Work Schedule\n\nDear [Manager's Name],\n\nI hope this email finds you well. I am writing to bring to your attention a couple of matters that require our team's collective consideration. Specifically, I would like to propose revisions to our company's dress code and work schedule.\n\nThe current dress code policy has been in place for some time now, and while it has served us well, I believe it is essential to update it to better reflect the evolving needs and expectations of our modern workplace. I suggest we adopt a more relaxed yet professional dress code that encourages employees to express their personal style while maintaining a level of professionalism.\n\nRegarding the dress code, I propose the following:\n\n- Business casual attire with a dress code that is not overly formal but still professional\n- A more lenient approach to jewelry and accessories\n- A dress code that is inclusi